In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.output_parsers import StrOutputParser


In [2]:

# Define the model ID for the Qwen language model we want to use from Hugging Face.
model_id = "meta-llama/Llama-3.1-8B-Instruct"

# You will need a Hugging Face token logged in via CLI for Llama 3.1
tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)
# Ensure pad_token exists (avoids generation warnings in many decoder-only models)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
    
model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    device_map="auto", # Automatically uses your GPU if available
    dtype="auto" # Optimizes memory usage
)

# --- 2. Create the Transformers Pipeline ---
# Note: For local pipelines, the task is strictly "text-generation"
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=220,
    temperature=0.1,
    return_full_text=False # Prevents the model from repeating your prompt
)

# --- 3. Wrap it for LangChain ---
# Convert the Hugging Face pipeline into a LangChain LLM
local_llm = HuggingFacePipeline(pipeline=pipe)

# CRITICAL: We STILL need ChatHuggingFace to format the history for Llama!
chat_model = ChatHuggingFace(llm=local_llm)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.
Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [3]:
ROLE_STOP_MARKERS = ("Human:", "System:", "AI:")

def parse_output(text: str) -> str:
    """Trim role-tag artifacts from generated text and return clean output."""
    cleaned = text
    for stop_word in ROLE_STOP_MARKERS:
        marker_index = cleaned.find(stop_word)
        if marker_index != -1:
            cleaned = cleaned[:marker_index]
            break
    return cleaned.strip()


### Short-term Buffer( sliding window )

In [4]:
# In-memory storage for different chat sessions.
store: dict[str, ChatMessageHistory] = {}

def get_session_history(session_id: str) -> ChatMessageHistory:
    """Return per-session chat history, creating it on first access."""
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]


### JIRA Assistant Chat Chain

### Function Overview
- `validate_jira_ticket_format`: checks required Jira headers and start line.
- `overwrite_last_ai_message`: syncs repaired output back into session memory.
- `validate_and_repair_jira`: runs validation + limited self-repair retries and updates repair stats.

In [5]:
# --- 3. Build the LCEL Chain ---
# We use MessagesPlaceholder to tell LangChain exactly where to inject the memory
# 1. Define your strict system prompt as a variable for cleaner code
system_instructions = """You are a strict Jira Ticket Formatter. 

RULES:
- Your goal is to help the user write detailed, well-structured Jira tickets (Bugs, Tasks, Stories, or Epics).
- Must include exactly these headers: Title:, Type:, Priority:, Description:, Labels:, Attachments:, Reporter:, Acceptance Criteria:.
- Be concise, direct, highly structured, and professional.
- REVISIONS: If the user asks to change or update something, you MUST rewrite the ENTIRE ticket from the previous turn. Apply their changes, but strictly keep all other previously established information intact. Do not erase previous details.

CRITICAL RULES:
- DO NOT ask follow-up questions.
- DO NOT output any multiple-choice options.
- DO NOT output ANY introductory, prefatory, or concluding text. 
- NEVER say things like "Here is the ticket" or acknowledge the user's prompt. 
- You must output ONLY the ticket content. Your response MUST start exactly with "Title:".
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_instructions),
    ("system", "Conversation summary (if any):\n{summary_context}"),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}")
])

jira_chain = prompt | chat_model | StrOutputParser() | parse_output

jira_chat_bot = RunnableWithMessageHistory(
    jira_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)

REPAIR_STATS = {
    "jira_repair": 0,
    "json_repair": 0,
}

REQUIRED_JIRA_HEADERS = [
    "Title:",
    "Type:",
    "Priority:",
    "Description:",
    "Labels:",
    "Attachments:",
    "Reporter:",
    "Acceptance Criteria:",
]

jira_repair_prompt = ChatPromptTemplate.from_messages([
    (
    "system",
    """You are a strict Jira ticket repair assistant.
        Fix the provided Jira ticket so it follows the required structure exactly.
        Rules:
        - Output only the ticket text.
        - Start with Title:.
        - Include all required headers exactly once.
        - Preserve user intent and existing facts.
        - Do not add unrelated details.
    """,
    ),
    (
        "human",
        "Required headers:\n{required_headers}\n\nUser intent:\n{user_input}\n\nDraft ticket:\n{ticket_text}\n\nRepaired ticket:",
    ),
])

jira_repair_chain = jira_repair_prompt | chat_model | StrOutputParser() | parse_output

def validate_jira_ticket_format(ticket_text: str) -> tuple[bool, list[str]]:
    """Validate Jira output structure and report missing required headers."""
    # Normalize lines and ignore blanks before structural checks.
    lines = [line.strip() for line in ticket_text.splitlines() if line.strip()]
    if not lines:
        return False, REQUIRED_JIRA_HEADERS.copy()

    missing_headers = [
        header for header in REQUIRED_JIRA_HEADERS
        if not any(line.startswith(header) for line in lines)
    ]

    starts_ok = lines[0].startswith("Title:")
    is_valid = starts_ok and not missing_headers
    return is_valid, missing_headers

def overwrite_last_ai_message(session_id: str, new_text: str) -> None:
    """Replace the most recent AI message in history with repaired content."""
    history = get_session_history(session_id)
    for message in reversed(history.messages):
        if getattr(message, "type", "") == "ai":
            message.content = new_text
            return

def validate_and_repair_jira(
    ticket_text: str,
    user_input: str,
    session_id: str,
    max_retries: int = 1,
    ) -> tuple[str, bool]:
    """Run format validation and apply limited self-repair passes if needed."""
    candidate = ticket_text
    repaired = False

    for attempt in range(max_retries + 1):
        # Exit early once candidate satisfies required structure.
        is_valid, _ = validate_jira_ticket_format(candidate)
        if is_valid:
            return candidate, repaired

        if attempt == max_retries:
            break

        required_headers_text = "\n".join(REQUIRED_JIRA_HEADERS)
        candidate = jira_repair_chain.invoke(
            {
                "required_headers": required_headers_text,
                "user_input": user_input,
                "ticket_text": candidate,
            }
        ).strip()
        repaired = True

    if repaired:
        REPAIR_STATS["jira_repair"] += 1
        overwrite_last_ai_message(session_id, candidate)

    return candidate, repaired

### Normal Chat Chain

In [6]:
normal_system_instructions = """You are a helpful and concise AI assistant.
- Answer clearly and directly.
- Ask follow-up questions only when needed to clarify user intent.
- Do not force Jira ticket formatting unless the user explicitly asks for Jira/template/ticket creation.
"""

normal_prompt = ChatPromptTemplate.from_messages([
    ("system", normal_system_instructions),
    ("system", "Conversation summary (if any):\n{summary_context}"),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}"),
])

normal_chain = normal_prompt | chat_model | StrOutputParser() | parse_output

normal_chat_bot = RunnableWithMessageHistory(
    normal_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)

### Judge Chain

don’t need a Judge yet for your current bot.
For your setup (normal chat + Jira routing), a single router is usually enough.
Add a Judge only if you see frequent wrong routing, low output quality, or need strict policy/safety checks.
A Judge adds extra latency, token cost, and complexity (it’s a second model call).



In [7]:

judge_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are a routing judge.
Given the user message and a tentative route, return exactly one final label: jira or normal.
Choose jira only if the user is clearly asking for Jira ticket/template creation or revision.
Otherwise return normal.
Do not output anything except jira or normal.""",
    ),
    (
        "human",
        "User message:\n{input}\n\nTentative route: {tentative_route}\n\nFinal route:",
    ),
])

judge_chain = judge_prompt | chat_model | StrOutputParser()

### Summary Memory Chain

### Function Overview
- `history_token_count`/`user_turn_count`: usage and turn counters.
- `normalize_text`: canonical text form for overlap checks.
- `get_recent_messages_text`: short-term memory snapshot.
- `get_summary_source_messages`: older history source for summarization.
- `remove_summary_overlap`: removes duplicated bullets.
- `maybe_refresh_summary`/`get_summary_context`: summary refresh and prompt context retrieval.

In [8]:
import re

SUMMARY_EVERY_TURNS = 5
SUMMARY_TOKEN_THRESHOLD = 1200
SHORT_TERM_BUFFER_MESSAGES = 12
SUMMARY_SOURCE_MAX_MESSAGES = 40
summary_store: dict[str, str] = {}

summary_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are a conversation memory assistant.
Update and compress the memory summary for one chat session.
Rules:
- Keep only durable facts, user preferences, decisions, constraints, and unresolved items.
- Remove small talk and duplicate details.
- Do not include details already present in the recent short-term buffer.
- Keep it concise (max 8 bullets).
- For Jira mode, preserve ticket facts (title intent, type, priority, labels, reporter, acceptance criteria decisions).
Return plain text bullet points only.""",
    ),
    (
        "human",
        "Route: {route}\n\nCurrent summary:\n{current_summary}\n\nLonger-term messages (excluding recent buffer):\n{summary_source_messages}\n\nUpdated summary:",
    ),
])

summary_chain = summary_prompt | chat_model | StrOutputParser()

def history_token_count(session_id: str) -> int:
    """Estimate token usage across the full message history of a session."""
    history = get_session_history(session_id)
    return sum(
        len(tokenizer.encode(str(getattr(msg, "content", "") or ""), add_special_tokens=False))
        for msg in history.messages
    )

def user_turn_count(session_id: str) -> int:
    """Count user-originated turns in a session history."""
    history = get_session_history(session_id)
    return sum(1 for msg in history.messages if getattr(msg, "type", "") == "human")

def normalize_text(text: str) -> str:
    """Normalize text for overlap checks using lowercase and punctuation stripping."""
    compact = re.sub(r"\s+", " ", text.strip().lower())
    return re.sub(r"[^a-z0-9 ]", "", compact)

def _messages_to_text(messages: list) -> str:
    """Render messages into role-prefixed plain text lines."""
    lines: list[str] = []
    for msg in messages:
        role = getattr(msg, "type", "unknown")
        content = str(getattr(msg, "content", "")).strip()
        lines.append(f"{role}: {content}")
    return "\n".join(lines)

def get_recent_messages_text(session_id: str, max_messages: int = SHORT_TERM_BUFFER_MESSAGES) -> str:
    """Serialize recent short-term window into role-prefixed plain text."""
    history = get_session_history(session_id)
    return _messages_to_text(history.messages[-max_messages:])

def get_summary_source_messages(
    session_id: str,
    skip_recent: int = SHORT_TERM_BUFFER_MESSAGES,
    max_messages: int = SUMMARY_SOURCE_MAX_MESSAGES,
    ) -> str:
    """Get older message text used as source when refreshing long-term summary."""
    history = get_session_history(session_id)
    older_messages = history.messages[:-skip_recent] if skip_recent > 0 else history.messages
    if not older_messages:
        return ""
    return _messages_to_text(older_messages[-max_messages:])

def remove_summary_overlap(summary_text: str, session_id: str) -> str:
    """Remove summary bullets that duplicate recent short-term buffer content."""
    if not summary_text.strip():
        return ""

    recent_corpus = normalize_text(get_recent_messages_text(session_id))
    if not recent_corpus:
        return summary_text.strip()

    cleaned_lines: list[str] = []
    for raw_line in summary_text.splitlines():
        line = raw_line.strip()
        if not line:
            continue

        bullet_body = line.lstrip("-*• ").strip()
        normalized = normalize_text(bullet_body)

        if normalized and len(normalized) >= 20 and normalized in recent_corpus:
            continue

        cleaned_lines.append(line if line.startswith(("-", "*", "•")) else f"- {line}")

    return "\n".join(cleaned_lines).strip()

def maybe_refresh_summary(session_id: str, route: str) -> None:
    """Refresh summary by cadence or token threshold, then de-duplicate overlap."""
    turns = user_turn_count(session_id)
    token_count = history_token_count(session_id)

    if turns == 0:
        return

    should_refresh = (turns % SUMMARY_EVERY_TURNS == 0) or (token_count >= SUMMARY_TOKEN_THRESHOLD)
    if not should_refresh:
        return

    current_summary = summary_store.get(session_id, "(none)")
    summary_source_messages = get_summary_source_messages(session_id)
    if not summary_source_messages.strip():
        return

    try:
        updated_summary = summary_chain.invoke(
            {
                "route": route,
                "current_summary": current_summary,
                "summary_source_messages": summary_source_messages,
            }
        ).strip()
        cleaned_summary = remove_summary_overlap(updated_summary, session_id)
        if cleaned_summary:
            summary_store[session_id] = cleaned_summary
    except Exception:
        pass

def get_summary_context(session_id: str) -> str:
    """Return cleaned summary text to inject into prompts as long-term context."""
    summary = summary_store.get(session_id, "")
    return remove_summary_overlap(summary, session_id)

### Router Chain

### Function Overview
- `track_metrics`: logs timestamped telemetry events.
- `get_metrics`/`clear_metrics`: query and reset telemetry.
- `keyword_route`: keyword fallback route selector.
- `detect_route_confidence`: heuristic confidence scorer.
- `parse_route_label`: normalizes route text and detects ambiguity.
- `detect_route_with_meta`/`detect_route`: main routing pipeline with judge/fallback metadata.

In [9]:
from typing import Any
import time

ROUTE_JIRA = "jira"
ROUTE_NORMAL = "normal"

JIRA_ROUTE_KEYWORDS = (
    "jira", "ticket", "story", "epic", "bug", "task", "acceptance criteria",
    "labels", "priority", "reporter", "attachments", "description",
)
JIRA_ROUTE_VERBS = ("create", "write", "draft", "update", "revise", "format", "extract")
NORMAL_ROUTE_MARKERS = ("explain", "what is", "how does", "summarize", "compare", "hello", "hi")

router_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are a routing classifier.
        Return exactly one label: jira or normal.
        Choose jira only when the user's message is asking to create, edit, revise, structure, or extract a Jira ticket/template.
        Choose normal for all other messages.
        Do not output anything except jira or normal.""",
    ),
    ("human", "{input}"),
])

router_chain = router_prompt | chat_model | StrOutputParser()
ROUTE_CONFIDENCE_THRESHOLD = 0.60

metrics_log: list[dict] = []

def track_metrics(event: str, payload: dict) -> None:
    """Store one telemetry event with timestamp for observability."""
    metrics_log.append({
        "timestamp": time.time(),
        "event": event,
        "payload": payload,
    })

def get_metrics(event: str | None = None, limit: int | None = None) -> list[dict]:
    """Return metrics, optionally filtered by event and limited to latest N."""
    filtered_items = metrics_log if event is None else [item for item in metrics_log if item.get("event") == event]
    if limit is None or limit <= 0:
        return filtered_items
    return filtered_items[-limit:]

def clear_metrics() -> None:
    """Remove all stored metrics events from memory."""
    metrics_log.clear()

def _count_hits(text: str, tokens: tuple[str, ...]) -> int:
    """Count how many token substrings are present in text."""
    return sum(1 for token in tokens if token in text)

def keyword_route(user_input: str) -> str:
    """Fallback route decision based on simple Jira keyword matching."""
    text = user_input.lower()
    return ROUTE_JIRA if _count_hits(text, JIRA_ROUTE_KEYWORDS) > 0 else ROUTE_NORMAL

def detect_route_confidence(text: str) -> float:
    """Heuristic confidence score for Jira routing in range [0.05, 0.95]."""
    lowered = text.lower()

    keyword_hits = _count_hits(lowered, JIRA_ROUTE_KEYWORDS)
    verb_hits = _count_hits(lowered, JIRA_ROUTE_VERBS)
    normal_hits = _count_hits(lowered, NORMAL_ROUTE_MARKERS)

    jira_signal = min(1.0, 0.18 * keyword_hits + 0.22 * verb_hits)
    normal_signal = min(1.0, 0.22 * normal_hits)

    confidence = 0.50 + (jira_signal - normal_signal) * 0.45
    return max(0.05, min(0.95, confidence))

def parse_route_label(raw: str) -> tuple[str, bool]:
    """Normalize model output to a route and flag ambiguous/invalid responses."""
    decision = (raw or "").strip().lower()
    has_jira = ROUTE_JIRA in decision
    has_normal = ROUTE_NORMAL in decision

    if has_jira and not has_normal:
        return ROUTE_JIRA, False
    if has_normal and not has_jira:
        return ROUTE_NORMAL, False
    if has_jira and has_normal:
        return ROUTE_NORMAL, True

    return ROUTE_NORMAL, True

def detect_route_with_meta(user_input: str) -> tuple[str, float, bool, bool, str]:
    """Select route with confidence-aware judge/fallback and return metadata + short reason."""
    judge_used = False
    fallback_used = False
    confidence = detect_route_confidence(user_input)
    reason = "router:uninitialized"

    try:
        router_raw = router_chain.invoke({"input": user_input})
        tentative_route, needs_judge = parse_route_label(router_raw)

        if needs_judge or confidence < ROUTE_CONFIDENCE_THRESHOLD:
            judge_used = True
            tentative_route = keyword_route(user_input)
            judge_raw = judge_chain.invoke({"input": user_input, "tentative_route": tentative_route})
            judged_route, still_ambiguous = parse_route_label(judge_raw)

            if still_ambiguous:
                fallback_used = True
                judged_route = keyword_route(user_input)
                reason = "route:fallback_after_ambiguous_judge"
            else:
                reason = "route:judge_selected_due_to_low_conf_or_ambiguity"
            route = judged_route
        else:
            route = tentative_route
            reason = "route:router_direct_high_confidence"

    except Exception as exc:
        fallback_used = True
        route = keyword_route(user_input)
        reason = f"route:exception_fallback:{type(exc).__name__}"

    track_metrics(
        "route_decision",
        {
            "route": route,
            "confidence": confidence,
            "judge_used": judge_used,
            "fallback_used": fallback_used,
            "reason": reason,
            "input_preview": user_input[:80],
        },
    )
    return route, confidence, judge_used, fallback_used, reason

def detect_route(user_input: str) -> str:
    """Convenience wrapper that returns only the final route label."""
    route, _, _, _, _ = detect_route_with_meta(user_input)
    return route


### Helper Functions

### Function Overview
- `validate_schema_payload`: enforces JSON schema shape via Pydantic.
- `convert_json`: extracts final Jira draft to JSON with one repair pass on failure.
- `JiraTemplateSchema`: target structured output fields for ticket conversion.

In [10]:
import json
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field
from typing import List

# --- 1. Define Your Target JSON Structure (The Parser) ---
class JiraTemplateSchema(BaseModel):
    title: str = Field(description="The title or summary of the Jira ticket")
    issue_type: str = Field(description="The type of the ticket (e.g., Bug, Task, Story, Epic)")
    priority: str = Field(description="The priority level (e.g., High, Medium, Low)")
    description: str = Field(description="The main description or body of the ticket")
    labels: List[str] = Field(description="A list of labels or tags associated with the ticket. If none, return an empty list.")
    attachments: str = Field(description="Details about any attachments, or 'None' if not applicable")
    reporter: str = Field(description="The name or placeholder of the person reporting the ticket")
    acceptance_criteria: str = Field(description="The criteria required for the ticket to be considered complete")

def validate_schema_payload(payload: dict) -> dict:
    """Validate extracted payload against schema and return normalized dict."""
    model_obj = JiraTemplateSchema.model_validate(payload)
    return model_obj.model_dump()

def _build_extraction_prompt(parser: JsonOutputParser) -> PromptTemplate:
    return PromptTemplate(
        template="""You are an expert data extraction algorithm.
        Extract the details from the provided Jira ticket text into JSON.
        Do not hallucinate or add extra information.

        {format_instructions}

        Ticket Text:
        {draft_text}
        """,
        input_variables=["draft_text"],
        partial_variables={"format_instructions": parser.get_format_instructions()},
    )

def _build_repair_prompt(parser: JsonOutputParser) -> PromptTemplate:
    return PromptTemplate(
        template="""You repair JSON output for a Jira schema.
        Return one valid JSON object only. No markdown and no extra text.

        {format_instructions}

        Ticket Text:
        {draft_text}
        """,
        input_variables=["draft_text"],
        partial_variables={"format_instructions": parser.get_format_instructions()},
    )

def convert_json(session_id: str = "jira_session_1") -> None:
    """Convert the latest Jira draft in session history to structured JSON."""
    conversion_start = time.perf_counter()
    user_history = get_session_history(session_id)

    if not user_history.messages:
        print("\nBot: No Jira draft found to convert.")
        track_metrics("json_conversion", {"session_id": session_id, "status": "no_draft"})
        return

    final_draft_text = user_history.messages[-1].content
    print("\nBot: Extracting your ticket into JSON...")

    parser = JsonOutputParser(pydantic_object=JiraTemplateSchema)
    extraction_chain = _build_extraction_prompt(parser) | chat_model | parser

    print("Extracting...\n")
    used_json_repair = False

    try:
        final_json_data = extraction_chain.invoke({"draft_text": final_draft_text})
    except Exception:
        print("Primary JSON parse failed. Running one repair pass...")
        REPAIR_STATS["json_repair"] += 1
        used_json_repair = True

        repair_chain = _build_repair_prompt(parser) | chat_model | StrOutputParser()
        repaired_raw = repair_chain.invoke({"draft_text": final_draft_text}).strip()

        try:
            repaired_payload = json.loads(repaired_raw)
            final_json_data = validate_schema_payload(repaired_payload)
        except Exception:
            print("JSON repair failed. Please continue refining the Jira draft and try again.")
            track_metrics(
                "json_conversion",
                {
                    "session_id": session_id,
                    "status": "failed",
                    "used_json_repair": used_json_repair,
                    "latency_ms": round((time.perf_counter() - conversion_start) * 1000, 2),
                },
            )
            return

    track_metrics(
        "json_conversion",
        {
            "session_id": session_id,
            "status": "success",
            "used_json_repair": used_json_repair,
            "latency_ms": round((time.perf_counter() - conversion_start) * 1000, 2),
        },
    )

    print("--- FINAL PARSED JSON ---")
    print(json.dumps(final_json_data, indent=2))

## Testing & Utility Functions

### Title: Quick Validation and Session Controls

Run `run_smoke_tests()` for quick checks, `get_memory_stats(session_id)` for diagnostics, and `reset_session(...)` to clear state.

### Function Overview
- `trim_history`: bounds message history and logs trim events.
- `get_memory_stats`: returns diagnostic memory/token stats.
- `reset_session`: clears one or all sessions with optional summary reset.
- `run_smoke_tests`: fast sanity assertions across routing/format/memory.
- `run_auto_test_suite`: orchestrates tests into a report dict.
- `print_failed_tests`: prints only failed test details for quick debugging.

### Title: Test Playbook Checklist

Use this checklist in order after running all setup/model cells.

- [ ] Run the utility code cell to register `run_smoke_tests`, `get_memory_stats`, and `reset_session`.
- [ ] Run `reset_session("all")` to start from a clean state.
- [ ] Run `run_smoke_tests()` and confirm all assertions pass.
- [ ] Run `get_memory_stats("jira_session_1")` and `get_memory_stats("normal_session_1")` to inspect token and summary usage.
- [ ] Start the main interface cell and test one normal prompt (expect non-Jira response).
- [ ] Test one Jira prompt (expect required Jira headers).
- [ ] Trigger approval (`looks good` / `approve`) and verify JSON extraction output appears.
- [ ] Confirm repair behavior by intentionally giving a malformed Jira request and checking `Repair stats` output increments only when needed.

Suggested Test Prompts

- Normal: `Explain OAuth in simple words.`
- Normal: `Summarize the pros and cons of microservices.`
- Jira: `Create a Jira bug ticket for login timeout after 30 seconds.`
- Jira revision: `Update priority to High and add acceptance criteria for retry behavior.`

In [11]:
def trim_history(session_id: str, keep_last_n: int = 40, clear_summary_if_trimmed: bool = False) -> int:
    """Trim session history to the latest N messages and return removed count."""
    if keep_last_n < 0:
        raise ValueError("keep_last_n must be >= 0")

    history = get_session_history(session_id)
    total_messages = len(history.messages)
    if total_messages <= keep_last_n:
        return 0

    if keep_last_n == 0:
        del history.messages[:]
    else:
        del history.messages[:-keep_last_n]

    removed = total_messages - len(history.messages)
    summary_cleared = False
    if clear_summary_if_trimmed and session_id in summary_store:
        del summary_store[session_id]
        summary_cleared = True

    track_metrics(
        "history_trim",
        {
            "session_id": session_id,
            "removed": removed,
            "remaining": len(history.messages),
            "keep_last_n": keep_last_n,
            "summary_cleared": summary_cleared,
        },
    )
    return removed

def get_memory_stats(session_id: str) -> dict:
    """Return quick memory diagnostics for history and summary storage."""
    history = get_session_history(session_id)
    summary_text = summary_store.get(session_id, "")

    return {
        "session_id": session_id,
        "message_count": len(history.messages),
        "user_turns": user_turn_count(session_id),
        "history_tokens": history_token_count(session_id),
        "summary_tokens": len(tokenizer.encode(summary_text, add_special_tokens=False)) if summary_text else 0,
        "summary_exists": bool(summary_text.strip()),
    }

def reset_session(session_id: str | None = None, clear_summary: bool = True) -> None:
    """Reset one session (or all) and optionally clear related summary memory."""
    if session_id is None or session_id == "all":
        store.clear()
        if clear_summary:
            summary_store.clear()
        track_metrics("session_reset", {"scope": "all", "clear_summary": clear_summary})
        print("All sessions cleared.")
        return

    store.pop(session_id, None)
    if clear_summary:
        summary_store.pop(session_id, None)

    track_metrics("session_reset", {"scope": session_id, "clear_summary": clear_summary})
    print(f"Session cleared: {session_id}")

def run_smoke_tests() -> None:
    """Run lightweight assertions for routing, Jira format checks, and memory helpers."""
    print("Running smoke tests...")

    route_cases = [
        ("Help me write a Jira bug ticket for login timeout", ROUTE_JIRA),
        ("Can you explain what OAuth is in simple terms?", ROUTE_NORMAL),
        ("Update the acceptance criteria and priority for this ticket", ROUTE_JIRA),
        ("Summarize our conversation in 3 bullets", ROUTE_NORMAL),
    ]
    for text, expected in route_cases:
        actual = detect_route(text)
        assert actual == expected, f"Route test failed: expected {expected}, got {actual} for '{text}'"

    confidence = detect_route_confidence("Please create a Jira bug ticket for login issue")
    assert 0.0 <= confidence <= 1.0, "Route confidence should be between 0 and 1"

    valid_ticket = """Title: Login error\nType: Bug\nPriority: High\nDescription: Users see 500 on login\nLabels: auth, backend\nAttachments: None\nReporter: QA Team\nAcceptance Criteria: Error fixed and tests pass"""
    invalid_ticket = """Type: Bug\nPriority: High\nDescription: Missing title and other fields"""
    assert validate_jira_ticket_format(valid_ticket)[0] is True, "Valid ticket should pass format check"
    assert validate_jira_ticket_format(invalid_ticket)[0] is False, "Invalid ticket should fail format check"

    test_session = "test_overlap_session"
    reset_session(test_session)
    history = get_session_history(test_session)
    history.add_user_message("Please keep priority high and reporter as QA team")
    history.add_ai_message("Noted. I will keep priority high and reporter as QA team")

    summary_candidate = "- Keep priority high and reporter as QA team\n- Unrelated long-term preference"
    cleaned = remove_summary_overlap(summary_candidate, test_session)
    assert "unrelated long-term preference" in cleaned.lower(), "Non-overlap summary line should remain"

    removed = trim_history(test_session, keep_last_n=1)
    assert removed >= 1, "trim_history should remove old messages when limit is exceeded"

    print("All smoke tests passed.")
    print(get_memory_stats(test_session))

In [12]:
def run_auto_test_suite() -> dict:
    """Execute the full local test bundle and return a structured report."""
    results: list[dict] = []

    def record(test_name: str, passed: bool, details: str = "") -> None:
        """Append one test outcome entry to the report list."""
        results.append({"test": test_name, "passed": passed, "details": details})

    try:
        reset_session("all")
        record("reset_all_sessions", True)
    except Exception as exc:
        record("reset_all_sessions", False, str(exc))

    try:
        run_smoke_tests()
        record("smoke_tests", True)
    except Exception as exc:
        record("smoke_tests", False, str(exc))

    try:
        jira_stats = get_memory_stats("jira_session_1")
        normal_stats = get_memory_stats("normal_session_1")
        required_keys = {"session_id", "message_count", "user_turns", "history_tokens", "summary_tokens", "summary_exists"}
        checks_ok = required_keys.issubset(jira_stats.keys()) and required_keys.issubset(normal_stats.keys())
        record("memory_stats_shape", checks_ok, f"jira={jira_stats}, normal={normal_stats}")
    except Exception as exc:
        record("memory_stats_shape", False, str(exc))

    try:
        sample = """Type: Bug\nPriority: High\nDescription: Missing fields"""
        repaired, used_repair = validate_and_repair_jira(
            ticket_text=sample,
            user_input="Please format this as a proper Jira ticket.",
            session_id="jira_session_1",
            max_retries=1,
        )
        is_valid, missing = validate_jira_ticket_format(repaired)
        record(
            "jira_self_repair",
            is_valid and used_repair,
            f"used_repair={used_repair}, missing={missing}, output_head={repaired[:120]}",
        )
    except Exception as exc:
        record("jira_self_repair", False, str(exc))

    total = len(results)
    passed = sum(1 for item in results if item["passed"])
    failed = total - passed

    print("=== AUTO TEST SUITE RESULTS ===")
    for item in results:
        mark = "PASS" if item["passed"] else "FAIL"
        print(f"[{mark}] {item['test']}")
        if item["details"]:
            print(f"      {item['details']}")
    print(f"Summary: {passed}/{total} passed, {failed} failed")

    return {"total": total, "passed": passed, "failed": failed, "results": results}

def print_failed_tests(report: dict) -> None:
    """Print only failed test cases from a `run_auto_test_suite` report."""
    failed_items = [item for item in report.get("results", []) if not item.get("passed", False)]
    if not failed_items:
        print("No failed tests.")
        return

    print("=== FAILED TESTS ===")
    for item in failed_items:
        print(f"- {item.get('test', 'unknown_test')}")
        details = item.get("details", "")
        if details:
            print(f"  details: {details}")

# Manual usage:
# suite_report = run_auto_test_suite()
# print_failed_tests(suite_report)


### Main Interface

### Interface Flow Overview
- Reads user input in a loop and supports `exit` to stop.
- Routes each turn via confidence-aware router (`jira` or `normal`).
- For `jira`, runs Jira chain, optional repair, summary refresh, and history trim.
- For `normal`, runs normal chain, summary refresh, and history trim.
- On approval keywords (`looks good`, `done`, `approve`), triggers JSON conversion from Jira draft.

In [13]:
jira_config: Any = {"configurable": {"session_id": "jira_session_1"}}
normal_config: Any = {"configurable": {"session_id": "normal_session_1"}}

EXIT_COMMANDS = {"exit"}
APPROVAL_COMMANDS = {"looks good", "im happy", "done", "approve"}
JIRA_SESSION_ID = "jira_session_1"
NORMAL_SESSION_ID = "normal_session_1"
HISTORY_KEEP_LAST_N = 40

# Reasoning summary controls (brief rationale only, not chain-of-thought).
SHOW_REASONING_TO_USER = True
MAX_REASONING_DISPLAY_CHARS = 160

def _format_reasoning_summary(
    route: str,
    route_confidence: float,
    route_reason: str,
    judge_used: bool,
    fallback_used: bool,
    ) -> str:
    """Build a short, user-friendly why/confidence summary."""
    mode_label = "Jira ticket mode" if route == ROUTE_JIRA else "Normal chat mode"
    decision_bits = []
    if judge_used:
        decision_bits.append("judge used")
    if fallback_used:
        decision_bits.append("keyword fallback")
    decision_suffix = f" ({', '.join(decision_bits)})" if decision_bits else ""

    reason_preview = (route_reason or "n/a")[:MAX_REASONING_DISPLAY_CHARS]
    return (
        f"Why: Selected {mode_label}{decision_suffix}. "
        f"Reason: {reason_preview}\n"
        f"Confidence: {route_confidence:.2f}"
    )

def _generate_jira_response(user_input: str) -> tuple[str, str, float, int]:
    """Handle one Jira-routed turn and return response, session id, latency, trim count."""
    session_id = JIRA_SESSION_ID
    summary_context = get_summary_context(session_id)

    start_ms = time.perf_counter()
    response = jira_chat_bot.invoke(
        {"input": user_input, "summary_context": summary_context},
        config=jira_config,
    )
    latency_ms = round((time.perf_counter() - start_ms) * 1000, 2)

    response, was_repaired = validate_and_repair_jira(
        ticket_text=response,
        user_input=user_input,
        session_id=session_id,
        max_retries=1,
    )
    if was_repaired:
        print("Bot: Applied one format self-repair pass for Jira output.")
        track_metrics("jira_repair", {"session_id": session_id, "triggered": True})

    maybe_refresh_summary(session_id, route=ROUTE_JIRA)
    trimmed = trim_history(session_id, keep_last_n=HISTORY_KEEP_LAST_N)
    return response, session_id, latency_ms, trimmed

def _generate_normal_response(user_input: str) -> tuple[str, str, float, int]:
    """Handle one normal-routed turn and return response, session id, latency, trim count."""
    session_id = NORMAL_SESSION_ID
    summary_context = get_summary_context(session_id)

    start_ms = time.perf_counter()
    response = normal_chat_bot.invoke(
        {"input": user_input, "summary_context": summary_context},
        config=normal_config,
    )
    latency_ms = round((time.perf_counter() - start_ms) * 1000, 2)

    maybe_refresh_summary(session_id, route=ROUTE_NORMAL)
    trimmed = trim_history(session_id, keep_last_n=HISTORY_KEEP_LAST_N)
    return response, session_id, latency_ms, trimmed

def _handle_approval() -> bool:
    """Run approval-to-JSON flow. Returns True when chat loop should stop."""
    jira_history = get_session_history(JIRA_SESSION_ID)
    if not jira_history.messages:
        print("\nBot: I don't have a Jira draft yet. Continue chatting or ask for a Jira ticket.")
        return False

    print("\nBot: Great! Moving Jira draft to JSON parsing phase...\n")
    convert_json(JIRA_SESSION_ID)
    return True

print("Bot: Hi! I can chat normally or help build Jira tickets. Tell me what you need.")


Bot: Hi! I can chat normally or help build Jira tickets. Tell me what you need.


In [14]:
while True:
    user_input = input("You: ").strip()
    if not user_input:
        continue

    normalized_input = user_input.lower()
    if normalized_input in EXIT_COMMANDS:
        break

    if normalized_input in APPROVAL_COMMANDS:
        if _handle_approval():
            break
        continue

    route, route_confidence, judge_used, fallback_used, route_reason = detect_route_with_meta(user_input)
    track_metrics(
        "turn_input",
        {
            "route": route,
            "route_confidence": round(route_confidence, 3),
            "judge_used": judge_used,
            "fallback_used": fallback_used,
            "route_reason": route_reason,
            "input_preview": user_input[:80],
        },
    )

    if SHOW_REASONING_TO_USER:
        reasoning_summary = _format_reasoning_summary(
            route=route,
            route_confidence=route_confidence,
            route_reason=route_reason,
            judge_used=judge_used,
            fallback_used=fallback_used,
        )
        print(f"Bot (reasoning summary): {reasoning_summary}")

    if route == ROUTE_JIRA:
        response, session_id, latency_ms, trimmed = _generate_jira_response(user_input)
    else:
        response, session_id, latency_ms, trimmed = _generate_normal_response(user_input)

    track_metrics(
        "response_generated",
        {
            "route": route,
            "session_id": session_id,
            "latency_ms": latency_ms,
            "trimmed_messages": trimmed,
            "route_reason": route_reason,
        },
    )

    print(f"\nBot:\n{response.strip()}\n")

print(f"Repair stats: {REPAIR_STATS}")
print(f"Metrics captured: {len(get_metrics())}")

Both `max_new_tokens` (=220) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=220) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=220) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Bot (reasoning summary): Why: Selected Normal chat mode (judge used). Reason: route:judge_selected_due_to_low_conf_or_ambiguity
Confidence: 0.50

Bot:
Here are 10 food options for you to consider:

1. Grilled chicken breast with roasted vegetables
2. Spaghetti Bolognese with garlic bread
3. Tacos with beef, lettuce, and avocado
4. Grilled salmon with quinoa and steamed asparagus
5. Veggie stir-fry with tofu and brown rice
6. Chicken Caesar salad
7. Burgers with sweet potato fries
8. Chicken fajitas with sautéed onions and bell peppers
9. Lentil soup with crusty bread
10. Grilled panini with ham and melted mozzarella cheese

Which type of cuisine or dietary preference do you have (e.g. vegetarian, gluten-free, etc.)?



Both `max_new_tokens` (=220) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=220) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=220) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Bot (reasoning summary): Why: Selected Jira ticket mode (judge used). Reason: route:judge_selected_due_to_low_conf_or_ambiguity
Confidence: 0.58


Both `max_new_tokens` (=220) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Bot (reasoning summary): Why: Selected Normal chat mode (judge used). Reason: route:judge_selected_due_to_low_conf_or_ambiguity
Confidence: 0.40

Bot:
My previous response contained the 10 food options.



Both `max_new_tokens` (=220) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=220) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=220) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Bot (reasoning summary): Why: Selected Normal chat mode (judge used). Reason: route:judge_selected_due_to_low_conf_or_ambiguity
Confidence: 0.50

Bot:
Here are the 10 food options I listed earlier:

1. Grilled chicken breast with roasted vegetables
2. Spaghetti Bolognese with garlic bread
3. Tacos with beef, lettuce, and avocado
4. Grilled salmon with quinoa and steamed asparagus
5. Veggie stir-fry with tofu and brown rice
6. Chicken Caesar salad
7. Burgers with sweet potato fries
8. Chicken fajitas with sautéed onions and bell peppers
9. Lentil soup with crusty bread
10. Grilled panini with ham and melted mozzarella cheese



Both `max_new_tokens` (=220) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Bot: Great! Moving Jira draft to JSON parsing phase...


Bot: Extracting your ticket into JSON...
Extracting...

--- FINAL PARSED JSON ---
{
  "title": "Checkout Page Crashes When Purchasing Assassin's Creed DLC",
  "issue_type": "Bug",
  "priority": "High",
  "description": "The checkout page crashes when attempting to purchase the new Assassin's Creed DLC. This issue prevents users from completing the purchase and accessing the DLC. Upon further investigation, it appears that the crash occurs when the user clicks the Complete Purchase button. The error message displayed is Internal Server Error: Unable to process transaction. The issue is reproducible on multiple browsers and devices.",
  "labels": [],
  "attachments": "None",
  "reporter": "Don John",
  "acceptance_criteria": ""
}
Repair stats: {'jira_repair': 0, 'json_repair': 0}
Metrics captured: 18
